# Auto-Label Notebook — Prompt Engineering & Face Cleaning
**AMD OnClick AI / ROCm 6.4 compatible**

Test prompt engineering for text generation and face clustering for album cleaning.

| # | Section | What you do |
|---|---------|-------------|
| 0 | Setup | Install packages |
| 1 | Load Data | Load 100 images from DownFlow/meizi |
| 2 | Prompt Eng | Interactive prompt engineering with Qwen |
| 3 | Face Detect | Test DeepFace face detection |
| 4 | Cluster | Test DBSCAN clustering |
| 5 | Pipeline | Full cleaning pipeline demo |

---
## 0 · Setup — Install Dependencies

> Run once per environment.

In [1]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Torch — AMD ROCm 6.4
pip('torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/rocm6.4')

# Core
pip('transformers>=5.0.0', 'huggingface_hub>=1.9.0',
    'datasets>=3.0.0', 'pandas>=2.0.0', 'Pillow>=10.0.0',
    'numpy>=1.24.0', 'scikit-learn>=1.4.0')

# Face detection
pip('facenet-pytorch', )

# YAML
pip('pyyaml>=6.0.0')

# YAML
pip('matplotlib')

# support CJK characters in matplotlib
import os
import urllib.request

font_url = "https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/SimplifiedChinese/NotoSansCJKsc-Regular.otf"
font_path = "NotoSansCJKsc-Regular.otf"

if not os.path.exists(font_path):
    urllib.request.urlretrieve(font_url, font_path)

from matplotlib import font_manager, rcParams
font_manager.fontManager.addfont(font_path)

prop = font_manager.FontProperties(fname=font_path)
font_name = prop.get_name()

rcParams["font.family"] = font_name
rcParams["axes.unicode_minus"] = False

print('Dependencies installed.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.10.1.dev395+g340ea86df.rocm641 requires depyf==0.19.0, but you have depyf 0.18.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.10.1.dev395+g340ea86df.rocm641 requires depyf==0.19.0, but you have depyf 0.18.0 which is incompatible.
vllm 0.10.1.dev395+g340ea86df.rocm641 requires setuptools<80,>=77.0.3; python_version > "3.11", but you have setuptools 82.0.1 which is incompatible.
vllm 0.10.1.dev395+g340ea86df.rocm641 requires setuptools<80.0.0,>=77.0.3, but you have setuptools 82.0.1 which is incompatible.
numba 0.61.2 requires nu

Dependencies installed.


---
## 1 · Load Data — Download from DownFlow/meizi

Load first 100 images from the dataset.

In [2]:
# Load dataset - first 100 rows
from datasets import load_dataset
from PIL import Image
import pandas as pd
import numpy as np
import io

DATASET_REPO = 'DownFlow/meizi'

print('Loading first 100 rows from dataset...')
ds = load_dataset(DATASET_REPO, split='train', streaming=True)
ds_sample = list(ds.take(100))
print(f'Loaded: {len(ds_sample)} images')

# Convert to dataframe
df = pd.DataFrame([{
    'image': x['image'],
    'file_name': x['file_name'],
    'album_id': x['album_id'],
    'title': x['title'],
    'model_name': x['model_name'],
    'tags': x['tags'],
    'text_en': x.get('text_en', ''),
    'text_cn': x.get('text_cn', ''),
} for x in ds_sample])

# Get unique albums
albums = df.groupby('album_id')
unique_album_count = len(albums)
print(f'Unique albums: {unique_album_count}')
print(f'Sample columns: {list(df.columns)}')

Loading first 100 rows from dataset...


Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

Loaded: 100 images
Unique albums: 99
Sample columns: ['image', 'file_name', 'album_id', 'title', 'model_name', 'tags', 'text_en', 'text_cn']


In [ ]:
# Show sample images
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for idx in range(min(10, len(df))):
    img = df.iloc[idx]['image']
    axes[idx].imshow(img)
    axes[idx].set_title(f"{df.iloc[idx]['album_id']}", fontsize=8)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f'Displayed first 10 images')

---
## 2 · Prompt Engineering

Use **huihui-ai/Huihui-Qwen3.5-2B-abliterated** for:
1. Generate text_en (English description)
2. Generate text_cn (Chinese description)
3. Score albums

Testing with 5 real images from the dataset.

In [8]:
# Load Qwen model for text generation
import torch
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration
MODEL_NAME = 'huihui-ai/Huihui-Qwen3.5-2B-abliterated'
#MODEL_NAME = 'huihui-ai/Huihui-Qwen3.5-4B-Claude-4.6-Opus-abliterated'
print('Loading processor...')
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print('Loading model (this may take a few minutes)...')
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype='auto',
    device_map='cuda',
    trust_remote_code=True
)
model.eval()

print(f'Model loaded: {MODEL_NAME}')

Loading tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Loading model (this may take a few minutes)...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Model loaded: huihui-ai/Huihui-Qwen3.5-2B-abliterated


In [9]:
# Generation function (text only)
def generate_text(prompt, max_tokens=256, temperature=0.7):
    """Generate text from text-only prompt."""
    messages = [{'role': 'user', 'content': prompt}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text], return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
        )
    
    # Decode (skipping the prompt tokens)
    prompt_len = inputs.input_ids.shape[1]
    response = processor.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    return response.strip()

# Generation function (with image) - using AutoProcessor
def generate_with_image(prompt, img_pil, max_tokens=256, temperature=0.7):
    """Generate text with image input using AutoProcessor."""
    # Standard message format with image tag
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image"}
            ],
        }
    ]
    
    # Processor handles the heavy lifting (PIL -> Tensors)
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img_pil], return_tensors='pt').to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=temperature,
        )
    
    # Decode (skipping the prompt tokens)
    prompt_len = inputs.input_ids.shape[1]
    response = processor.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    return response.strip()

print('Generation functions ready (text + image).')

Generation functions ready (text + image).


In [ ]:
# Test prompt: text_en with 5 real images (using base64 image)
import matplotlib.pyplot as plt

# Prompts to test
# prompt_en = "Describe this image in detail. Include: subject appearance, pose, clothing, environment, lighting, and mood. Output 2-3 sentences in English."
prompt_en = "You are an admin of a porn website. You are judging asia women pictures in terms of appearance, pose, closing, environment, lighting, and mood. Check if the picture is arousing and give clear prompt for reconstructing the image"
# Test with 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 8))
axes = axes.flatten()

results_en = []
for idx in range(5):
    # Get real image
    img = df.iloc[idx]['image']
    axes[idx].imshow(img)
    axes[idx].axis('off')
    
    # Generate with real image
    result = generate_with_image(prompt_en, img, max_tokens=128)
    results_en.append(result)
    
    # Show result in title (truncated)
    axes[idx].set_title(f"{result[:50]}...", fontsize=7)

plt.suptitle('Test Prompt: text_en with Real Images', fontsize=12)
plt.tight_layout()
plt.show()

print('\n=== Generated English Descriptions ===')
for i, result in enumerate(results_en):
    print(f'\nImage {i+1}:')
    print(f'  {result}')

In [ ]:
# Test prompt: text_cn with 5 real images (using base64 image)
import matplotlib.pyplot as plt

# Prompt in Chinese
prompt_cn = "你是一个老色鬼，详细描述图片。"

# Test with 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 8))
axes = axes.flatten()

results_cn = []
for idx in range(5):
    img = df.iloc[idx]['image']
    axes[idx].imshow(img)
    axes[idx].axis('off')
    
    result = generate_with_image(prompt_cn, img, max_tokens=128)
    results_cn.append(result)
    
    axes[idx].set_title(f"{result[:30]}...", fontsize=7)

plt.suptitle('Test Prompt: text_cn (中文描述)', fontsize=12)
plt.tight_layout()
plt.show()

print('\n=== Generated Chinese Descriptions ===')
for i, result in enumerate(results_cn):
    print(f'\nImage {i+1}:')
    print(f'  {result}')

In [ ]:
# Test prompt: scoring with 5 real images (using base64 image)
import matplotlib.pyplot as plt

# Scoring prompt
prompt_score = "Rate this image quality from 1-10 (1=poor, 10=excellent). Consider: composition, lighting, subject clarity, background. Output only a single number."

# Test with 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 8))
axes = axes.flatten()

results_score = []
for idx in range(5):
    img = df.iloc[idx]['image']
    axes[idx].imshow(img)
    axes[idx].axis('off')
    
    result = generate_with_image(prompt_score, img, max_tokens=32, temperature=0.3)
    results_score.append(result)
    
    axes[idx].set_title(f"Score: {result[:20]}", fontsize=8)

plt.suptitle('Test Prompt: Scoring (1-10)', fontsize=12)
plt.tight_layout()
plt.show()

print('\n=== Generated Scores ===')
for i, result in enumerate(results_score):
    print(f'Image {i+1}: {result}')

---
### Save Best Prompts

After testing, save the best prompts to config/prompts.yaml

In [ ]:
# Save best prompts
import yaml
import os

best_prompts = {
    'text_en': {
        'system': 'You are an image captioning expert.',
        'template': "You are an image captioning expert. Generate a detailed English description (2-3 sentences) describing the subject, environment, and style. Image: {description}. Description:"
    },
    'text_cn': {
        'system': '你是一个图像描述专家。',
        'template': '你是一个图像描述专家。用中文详细描述这张图片（2-3句），包括主体、环境、风格。图片：{description}。描述：'
    },
    'scoring': {
        'system': 'You are a photography quality expert.',
        'template': '作为摄影质量专家，评估这张照片质量（1-10分）。评判标准：面部清晰度(30%)、光照质量(25%)、构图(25%)、整体美感(20%)。仅输出数字。图片：{description}。分数：'
    }
}

os.makedirs('config', exist_ok=True)
with open('config/prompts.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(best_prompts, f, allow_unicode=True, default_flow_style=False)

print('Saved best prompts to config/prompts.yaml')

---
## 3 · Face Detection with DeepFace

Test DeepFace for face detection and embedding extraction.

**ROCm Tip:** Set `HIP_VISIBLE_DEVICES=0` to ensure GPU usage.

In [ ]:
# Setup FaceNet (MTCNN + InceptionResnet)
import torch
import os
os.environ['HIP_VISIBLE_DEVICES'] = '0'  # Use first GPU

from facenet_pytorch import MTCNN, InceptionResnetV1
import numpy as np
from PIL import Image

# Initialize MTCNN for face detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(
    image_size=160,
    margin=0,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=True,
    device=device,
    keep_all=True
)

# Initialize InceptionResnet for face embeddings
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

print('FaceNet loaded (MTCNN + InceptionResnetV1).')


In [ ]:
# Test face detection with 5 real images using MTCNN
import matplotlib.pyplot as plt
import cv2
import numpy as np
from torchvision import transforms

def detect_and_draw(img_pil):
    """Face detection using MTCNN."""
    # Convert PIL to RGB numpy
    img_np = np.array(img_pil)
    
    try:
        # Detect faces with MTCNN
        boxes, probs = mtcnn.detect(img_np)
        
        # Draw bounding boxes
        img_cv = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        face_count = 0
        if boxes is not None:
            for box, prob in zip(boxes, probs):
                if prob > 0.8:  # Confidence threshold
                    x1, y1, x2, y2 = box.astype(int)
                    cv2.rectangle(img_cv, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    face_count += 1
        
        return cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB), face_count
    except Exception as e:
        print(f'Error: {e}')
        return img_pil, 0

# Test with 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 8))
axes = axes.flatten()

face_counts = []
for idx in range(5):
    img = df.iloc[idx]['image']
    img_drawn, count = detect_and_draw(img)
    face_counts.append(count)
    
    axes[idx].imshow(img_drawn)
    axes[idx].set_title(f'Faces: {count}', fontsize=10)
    axes[idx].axis('off')

plt.suptitle('Face Detection with MTCNN', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Face counts: {face_counts}')

In [ ]:
# Test face embeddings with 5 images using InceptionResnetV1
import matplotlib.pyplot as plt

# Test embedding extraction with 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 8))
axes = axes.flatten()

embeddings = []

for idx in range(5):
    img = df.iloc[idx]['image']
    axes[idx].imshow(img)
    axes[idx].axis('off')
    
    try:
        # Detect and crop face
        img_cropped = mtcnn(img)
        
        if img_cropped is not None:
            # Handle multiple faces - take the first one
            if img_cropped.dim() == 4:
                img_cropped = img_cropped[0]
            
            # Calculate embedding
            img_embedding = resnet(img_cropped.unsqueeze(0).to(device))
            embeddings.append(img_embedding.detach().cpu().numpy()[0])
            axes[idx].set_title(f'Emb: {len(embeddings[-1])}D', fontsize=9)
        else:
            embeddings.append(None)
            axes[idx].set_title('No face', fontsize=9)
    except Exception as e:
        print(f'Error: {e}')
        embeddings.append(None)
        axes[idx].set_title('Error', fontsize=9)

plt.suptitle('Face Embeddings (InceptionResnetV1)', fontsize=12)
plt.tight_layout()
plt.show()

valid_embs = [e for e in embeddings if e is not None]
print(f'Extracted {len(valid_embs)} embeddings')
if valid_embs:
    print(f'Embedding dimension: {len(valid_embs[0])}')

---
## 4 · DBSCAN Clustering

Use DBSCAN to cluster face embeddings and identify the target person.

- **eps**: Similarity threshold (~0.35-0.45 for ArcFace cosine distance)
- **min_samples**: Minimum photos to form a cluster (~3-5)

In [ ]:
# Test DBSCAN with real embeddings
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_distances
import numpy as np

if len(valid_embs) >= 3:
    # Convert to array
    emb_matrix = np.array(valid_embs)
    
    # Compute cosine distance
    distance_matrix = cosine_distances(emb_matrix)
    
    # DBSCAN
    eps = 0.4
    min_samples = 2
    
    clustering = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
    labels = clustering.fit_predict(distance_matrix)
    
    print(f'eps={eps}, min_samples={min_samples}')
    print(f'Labels: {labels}')
    print(f'Unique clusters: {set(labels)}')
    
    # Show results
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, len(valid_embs), figsize=(15, 3))
    if len(valid_embs) == 1:
        axes = [axes]
    
    for idx in range(len(valid_embs)):
        img = df.iloc[idx]['image']
        axes[idx].imshow(img)
        axes[idx].set_title(f'Cluster: {labels[idx]}', fontsize=10)
        axes[idx].axis('off')
    
    plt.suptitle('DBSCAN Clustering Results', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough embeddings for clustering, using synthetic data instead...')
    
    # Generate synthetic embeddings for demo
    np.random.seed(42)
    embedding_dim = 512
    
    # Alice cluster (3 samples)
    alice_centroid = np.random.randn(embedding_dim)
    alice_centroid = alice_centroid / np.linalg.norm(alice_centroid)
    alice_embs = alice_centroid + np.random.randn(3, embedding_dim) * 0.1
    alice_embs = alice_embs / np.linalg.norm(alice_embs, axis=1, keepdims=True)
    
    # Noise (2 samples)
    noise_embs = np.random.randn(2, embedding_dim)
    noise_embs = noise_embs / np.linalg.norm(noise_embs, axis=1, keepdims=True)
    
    demo_embs = np.vstack([alice_embs, noise_embs])
    distance_matrix = cosine_distances(demo_embs)
    
    clustering = DBSCAN(eps=0.4, min_samples=2, metric='precomputed')
    labels = clustering.fit_predict(distance_matrix)
    
    print(f'Demo with synthetic data:')
    print(f'Labels: {labels}')

---
## 5 · Full Cleaning Pipeline

**Phase 2: The Cleaning Pipeline**

For each album folder:
1. **Face Extraction & Embedding**: Scan every image, detect faces, convert to embeddings
2. **Clustering**: Use DBSCAN to group similar faces
3. **Identify Target**: Find the largest cluster (assumed to be the labeled person)
4. **Purge**: Move images outside the target cluster to Removed/ folder

In [ ]:
# Full cleaning pipeline using MTCNN + InceptionResnetV1
import os
import shutil
from pathlib import Path

def clean_album(album_path, eps=0.4, min_samples=3):
    """
    Clean an album folder using MTCNN + InceptionResnetV1 face clustering.
    
    Args:
        album_path: Path to album folder
        eps: DBSCAN epsilon (cosine distance threshold)
        min_samples: Minimum samples per cluster
    
    Returns:
        Dict with cleaning results
    """
    album_path = Path(album_path)
    removed_path = album_path / 'Removed'
    removed_path.mkdir(exist_ok=True)
    
    # Step 1: Extract face embeddings
    print('Step 1: Extracting face embeddings...')
    embeddings = []
    image_paths = []
    
    for img_path in album_path.glob('*.jpg'):
        try:
            from PIL import Image
            img = Image.open(img_path).convert('RGB')
            
            # Detect and crop face
            img_cropped = mtcnn(img)
            
            if img_cropped is not None:
                # Handle multiple faces - take the first one
                if img_cropped.dim() == 4:
                    img_cropped = img_cropped[0]
                
                # Calculate embedding
                emb = resnet(img_cropped.unsqueeze(0).to(device))
                embeddings.append(emb.detach().cpu().numpy()[0])
                image_paths.append(img_path)
        except Exception as e:
            print(f'  Skipped {img_path.name}: {e}')
    
    if not embeddings:
        print('No faces detected!')
        return {'kept': 0, 'removed': 0}
    
    embeddings = np.array(embeddings)
    print(f'  Extracted {len(embeddings)} embeddings')
    
    # Step 2: Clustering
    print('Step 2: Clustering with DBSCAN...')
    from sklearn.cluster import DBSCAN
    from sklearn.metrics.pairwise import cosine_distances
    
    distance_matrix = cosine_distances(embeddings)
    clustering = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
    labels = clustering.fit_predict(distance_matrix)
    
    # Step 3: Identify target
    print('Step 3: Identifying target person...')
    cluster_counts = {}
    for label in set(labels):
        if label != -1:
            cluster_counts[label] = list(labels).count(label)
    
    if not cluster_counts:
        print('  No clusters found!')
        return {'kept': 0, 'removed': len(image_paths)}
    
    target_label = max(cluster_counts, key=cluster_counts.get)
    target_count = cluster_counts[target_label]
    print(f'  Target cluster: {target_count} images')
    
    # Step 4: Purge
    print('Step 4: Purging non-target images...')
    kept_count = 0
    removed_count = 0
    
    for i, img_path in enumerate(image_paths):
        if labels[i] == target_label:
            kept_count += 1
        else:
            dest = removed_path / img_path.name
            shutil.move(str(img_path), str(dest))
            removed_count += 1
    
    print(f'  Done: {kept_count} kept, {removed_count} moved to Removed/')
    return {'kept': kept_count, 'removed': removed_count}

# Example usage:
# result = clean_album('/path/to/album_folder')
print('clean_album() function ready.')

In [ ]:
print('Notebook complete!')
# Summary:
# 1. Loaded 100 real images from DownFlow/meizi
# 2. Tested Qwen prompts with 5 images each
# 3. Tested DeepFace face detection
# 4. Tested DBSCAN clustering
# 5. Full cleaning pipeline ready
# 
# Next: Update config/prompts.yaml, run on actual albums